<a href="https://colab.research.google.com/github/Honorine-Kabore/genomic_mosq/blob/main/NJT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q --no-warn-conflicts malariagen_data

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 30.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.8/215.8 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.7/71.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.9/775.9 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.3/211.3 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 M

In [2]:
import malariagen_data
import allel
import numpy as np
import plotly.express as px
import plotly.io as pio
import pandas as pd

In [3]:
# plotting setup
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.gridspec import GridSpec
import matplotlib_venn as venn

In [4]:
#Mounting Google Drive
import os
from google.colab import drive
drive.mount("drive")

# make dir
results_dir = "drive/MyDrive/Genomic/"
os.makedirs(results_dir, exist_ok=True)

Mounted at drive


In [5]:
ag3 = malariagen_data.Ag3(pre=True, results_cache=results_dir)
ag3

<MalariaGEN Ag3 API client>
Storage URL                           : gs://vo_agam_release_master_us_central1/
Data releases available               : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16, 3.17
Results cache                         : /content/drive/MyDrive/Genomic
Cohorts analysis                      : 20260120
AIM analysis                          : 20220528
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.6.0
Client location                       : Nevada, United States (Google Cloud us-west4)
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16, 3.17
---
Please note that data are subject to terms of use,
for more information see https://www.malariagen.net/data
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.6.0/Ag3.html

In [6]:
sets =['3.7', '3.8','3.9', '3.10','3.11']
df_samples = ag3.sample_metadata(sample_sets=sets)
sample_query='country=="Burkina Faso"'


In [ ]:
bf_samples = df_samples.query('country == "Burkina Faso" and year > 2004')

In [7]:
# Create a new cohort for taxon
bf_samples = df_samples.query('country == "Burkina Faso" and year > 2004')
cohort_ta = {}
for ta in bf_samples.taxon.unique():
 for lo in bf_samples.location.unique():
  sample_list = list(bf_samples.query("taxon==@ta").sample_id)
  cohort_ta['An.'+ta] = "sample_id in ['"+ "', '".join(sample_list) + "']"

In [8]:
#create a new cohort with location, taxon
bf_samples = df_samples.query('country == "Burkina Faso" and year > 2004')
cohort_re = {}
for ta in bf_samples.taxon.unique():
    for re in bf_samples.admin1_name.unique():
        sample_list = list(bf_samples.query("admin1_name==@re and taxon==@ta").sample_id)
        cohort_re[re + "_" + ta[:4]] = "sample_id in ['"+ "', '".join(sample_list) + "']"

# NJT

In [9]:
ag3.plot_njt(
    region="3L:15,000,000-41,000,000",
    sample_sets=sets,
    sample_query= "country=='Burkina Faso'and year > 2004",
    n_snps=30_000,
    color="taxon",
    #symbol="location",
    min_cohort_size= 10,
    edge_legend=False,
    marker_size=10,
)

Compute SNP allele counts:   0%|          | 0/15051 [00:00<?, ?it/s]

Compute biallelic diplotypes:   0%|          | 0/16513 [00:00<?, ?it/s]

Construct neighbour-joining tree:   0%|          | 0/1085 [00:00<?, ?it/s]

In [ ]:
ag3.plot_njt(
    region="3L:15,000,000-41,000,000",
    sample_sets=sets,
    sample_query= "country=='Burkina Faso'",
    n_snps=100_000,
    color="taxon",
    #symbol="location",
    min_cohort_size= 10,
    edge_legend=False,
    marker_size=10,
)

In [10]:
#arabiensis
ag3.plot_njt(
    region="3L:15,000,000-41,000,000",
    sample_sets=sets,
    sample_query= "country=='Burkina Faso' and taxon=='arabiensis' and year > 2004",
    n_snps=30_000,
    color="admin1_name",
    #symbol="taxon",
    edge_legend=False,
    marker_size=10,)

Compute SNP allele counts:   0%|          | 0/6795 [00:00<?, ?it/s]

Compute biallelic diplotypes:   0%|          | 0/7139 [00:00<?, ?it/s]

Construct neighbour-joining tree:   0%|          | 0/131 [00:00<?, ?it/s]

In [11]:
#arabienis
ag3.plot_njt(
    region="3L:15,000,000-41,000,000",
    sample_sets=sets,
    sample_query= "country=='Burkina Faso' and taxon=='arabiensis'and year > 2004",
    n_snps=40_000,
    color="admin1_name",
    #symbol="taxon",
    edge_legend=False,
    marker_size=10,
)

Compute SNP allele counts:   0%|          | 0/6451 [00:00<?, ?it/s]

Compute biallelic diplotypes:   0%|          | 0/6795 [00:00<?, ?it/s]

Construct neighbour-joining tree:   0%|          | 0/127 [00:00<?, ?it/s]

In [13]:
#coluzzii
ag3.plot_njt(
    region="3L:15,000,000-41,000,000",
    sample_sets=sets,
    sample_query= "country=='Burkina Faso' and taxon=='coluzzii' and year > 2004",
    n_snps=30_000,
    color="location",
    #symbol="taxon",
    edge_legend=False,
    marker_size=10,
)

Compute SNP allele counts:   0%|          | 0/13159 [00:00<?, ?it/s]

Compute biallelic diplotypes:   0%|          | 0/14277 [00:00<?, ?it/s]

Construct neighbour-joining tree:   0%|          | 0/730 [00:00<?, ?it/s]

In [14]:
#coluzzii
ag3.plot_njt(
    region="3L:15,000,000-41,000,000",
    sample_sets=sets,
    sample_query= "country=='Burkina Faso' and taxon=='coluzzii' and year > 2004",
    n_snps=30_000,
    color="admin1_name",
    #symbol="taxon",
    edge_legend=False,
    marker_size=10,
)

In [15]:
#gambiae
ag3.plot_njt(
    region="3L:15,000,000-41,000,000",
    sample_sets=sets,
    sample_query= "country=='Burkina Faso' and taxon=='gambiae' and year > 2004",
    n_snps=30_000,
    color="admin1_name",
    #symbol="taxon",
    edge_legend=False,
    marker_size=10,
)

Compute SNP allele counts:   0%|          | 0/7741 [00:00<?, ?it/s]

Compute biallelic diplotypes:   0%|          | 0/8171 [00:00<?, ?it/s]

Construct neighbour-joining tree:   0%|          | 0/165 [00:00<?, ?it/s]

In [ ]:
#x chrromsosome
ag3.plot_njt(
    region="X",
    sample_sets=sets,
    sample_query= "country=='Burkina Faso'and year > 2004",
    n_snps=30_000,
    color="taxon",
    #symbol="taxon",
    edge_legend=False,
    marker_size=10,)

In [16]:
#arabiensis X chromosome
ag3.plot_njt(
    region="X",
    sample_sets=sets,
    sample_query= "country=='Burkina Faso' and taxon=='arabiensis'and year > 2004",
    n_snps=30_000,
    color="admin1_name",
    #symbol="taxon",
    edge_legend=False,
    marker_size=10,
)

Compute SNP allele counts:   0%|          | 0/4681 [00:00<?, ?it/s]

Compute biallelic diplotypes:   0%|          | 0/4993 [00:00<?, ?it/s]

Construct neighbour-joining tree:   0%|          | 0/131 [00:00<?, ?it/s]

In [19]:
#coluzzii X
ag3.plot_njt(
    region="X",
    sample_sets=sets,
    sample_query= "country=='Burkina Faso' and taxon=='coluzzii'and year > 2004",
    n_snps=30_000,
    color="admin1_name",
    #symbol="taxon",
    edge_legend=False,
    marker_size=10,)

Compute SNP allele counts:   0%|          | 0/9829 [00:00<?, ?it/s]

Compute biallelic diplotypes:   0%|          | 0/10843 [00:00<?, ?it/s]

Construct neighbour-joining tree:   0%|          | 0/730 [00:00<?, ?it/s]

In [17]:
#chromosome X gambiae
ag3.plot_njt(
    region="X",
    sample_sets=sets,
    sample_query= "country=='Burkina Faso' and taxon=='gambiae'and year > 2004",
    n_snps=30_000,
    color="admin1_name",
    #symbol="taxon",
    edge_legend=False,
    marker_size=10,)

Compute SNP allele counts:   0%|          | 0/5383 [00:00<?, ?it/s]

Compute biallelic diplotypes:   0%|          | 0/5773 [00:00<?, ?it/s]

Construct neighbour-joining tree:   0%|          | 0/165 [00:00<?, ?it/s]